Generate Collect Earth Online samples for the country plots made in `Generate_Plots_For_Countries.ipynb`, following the
CEO sample CSV example (sample distribution csv). Each plot is a 10x10 pixel block of the 30m grid and its samples are 20
randomly chosen pixel centres in the block, so samples align with the GeoMAD and LULC pixels.

1. Read each country plots GeoJSON.
2. Take the 10x10 pixel centres of each plot block in the grid CRS for the country's region.
3. Randomly choose the samples per plot.
4. Write a CSV per country with PLOTID, SAMPLEID, LAT and LON (EPSG:4326).

In [13]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

from ldn.grids import get_gridspec
from ldn.utils import DEP_COUNTRIES_AND_CODES, WGS84, get_analysis_epsg

In [14]:
PLOTS_DIR = Path("country_plots")
SAMPLES_DIR = Path("country_samples")
SAMPLES_DIR.mkdir(exist_ok=True)
SAMPLES_PER_PLOT = 20
PLOT_PIXELS = 10  # Must match the plots notebook
SEED = 0

In [15]:
def make_plot_samples(plots_path: Path) -> pd.DataFrame:
    """Return randomly chosen pixel centre samples inside each plot.

    Args:
        plots_path: Country plots GeoJSON, named after the country.

    Returns:
        DataFrame with PLOTID, SAMPLEID (1 to SAMPLES_PER_PLOT within each plot), LAT and LON (EPSG:4326).
    """
    plots = gpd.read_file(plots_path)
    region = "pacific" if plots_path.stem in DEP_COUNTRIES_AND_CODES else "non-pacific"
    analysis_crs = get_analysis_epsg(region)
    res = get_gridspec(region).resolution.x
    centres = plots.to_crs(analysis_crs).centroid
    rng = np.random.default_rng(SEED)

    # Pixel centre offsets from the plot centre
    offsets = (np.arange(PLOT_PIXELS) - (PLOT_PIXELS - 1) / 2) * res
    dx, dy = (a.ravel() for a in np.meshgrid(offsets, offsets))

    rows = []
    for plot_id, centre in zip(plots["PLOTID"], centres):
        pick = rng.choice(len(dx), size=SAMPLES_PER_PLOT, replace=False)
        points = gpd.GeoSeries(gpd.points_from_xy(centre.x + dx[pick], centre.y + dy[pick]), crs=analysis_crs)
        points = points.to_crs(WGS84)
        rows.append(
            pd.DataFrame(
                {"PLOTID": plot_id, "SAMPLEID": np.arange(1, SAMPLES_PER_PLOT + 1), "LAT": points.y, "LON": points.x}
            )
        )
    print(f"{plots_path.stem}: {len(plots)} plots, {SAMPLES_PER_PLOT} samples each")

    return pd.concat(rows, ignore_index=True)

In [16]:
for plots_path in sorted(PLOTS_DIR.glob("*.geojson")):
    make_plot_samples(plots_path).to_csv(SAMPLES_DIR / f"{plots_path.stem}_samples.csv", index=False)

Anguilla: 10 plots, 20 samples each
Antigua and Barbuda: 10 plots, 20 samples each
Aruba: 10 plots, 20 samples each
